In [ ]:
# Install the required libraries with specific versions for compatibility.

# Transformers: Provides pre-trained language models.
# Hugging Face Hub: Downloads models from the Hugging Face Hub.
# Tokenizers: Converts text into tokens for the model.
# Accelerate: Improves model execution on CPU/GPU.
# SentencePiece: Required by models like T5 and LLaMA.
# Safetensors: Loads model weights safely and efficiently.

!pip install \
transformers==4.46.3 \
huggingface_hub==0.26.2 \
tokenizers==0.20.3 \
accelerate==1.1.1 \
sentencepiece \
safetensors

In [ ]:
# Install BERTopic version 0.16.4.
# BERTopic is used to automatically discover and analyze topics in a collection of text documents.

!pip install bertopic==0.16.4

In [ ]:
# Load data from Hugging Face
from datasets import load_dataset
# load the dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]
# Extract abstract column from the dataset
abstracts = dataset["Abstracts"]
# Extrat titles from the dataset
titles = dataset["Titles"]

In [ ]:
# Prints the length of abstracts column
print(len(abstracts))

In [ ]:
#Imports sentence transformers from sentence_transformers library
from sentence_transformers import SentenceTransformer
# Load the ore trained embedding model to convert to text int embeddings
embedding_model = SentenceTransformer("thenlper/gte-small")
# Convert all the abstract text into embeddings
embeddings = embedding_model.encode(
    abstracts,
    # Number of abstracts processed at one time
    batch_size=256,          # Try 256
    # show the progress bar
    show_progress_bar=True,
    # Return the embeddings as a NumPy array instead of a Python list.
    # NumPy arrays are faster and easier to use for mathematical operations.
    convert_to_numpy=True
)


In [ ]:
# Check the dimensions of the resulting embeddings
embeddings.shape

In [ ]:
# Imports UMAP classs from umap library
#  UMAP is a dimensionality reduction technique that reduces high-dimensional data into fewer dimensions while preserving the relationships between similar data points.
from umap import UMAP
# We reduce the input embeddings from 384 dimensions to 5 dimensions
umap_model = UMAP(
#n_compnents = number of new components
# min_dist = Controls how closely UMAP packs similar data points together.
# A value of 0.0 keeps similar points as close as possible.
#  Use cosine distance to measure similarity between embeddings. Cosine distance works well for text embeddings   beacause it compare meaning for similarity
n_components=5, min_dist=0.0, metric='cosine',
# Fix the random seed so that output may same evry time we execute the code
random_state=42
)
# Fit or train the UMAP model to the embeddings and transform them.
# This learns the structure of the original 384-dimensional embeddings and converts them into 5-dimensional embeddings.
reduced_embeddings = umap_model.fit_transform(embeddings)

In [ ]:
# Import HDBSCAN class from hdbscan library
# HDBSCAN is used to group similar embeddings into cluster
from hdbscan import HDBSCAN
# create and train the HDBSCAN model
# We fit the model and extract the clusters
hdbscan_model = HDBSCAN(
# Minimum number of data points required to form a cluster
# uses euclidean method to measure the distance between the embeddings
min_cluster_size=50, metric="euclidean",
# "eom" (Excess of Mass) is the default cluster selection method.
# It chooses the most stable and meaningful clusters automatically.
cluster_selection_method="eom"
).fit(reduced_embeddings)
clusters = hdbscan_model.labels_
# How many clusters did we generate?
len(set(clusters))

In [ ]:
# import numpy
import numpy as np
# Print first three documents in cluster 0
# np.where(clusters == cluster) returns the positions where the cluster label is equal to 0.
# Print the first 300 characters of each abstract.
cluster = 0
for index in np.where(clusters==cluster)[0][:3]:print(abstracts[int(index)][:300] + "... \n")

In [ ]:
# import pandas
import pandas as pd
# Reduce 384-dimensional embeddings to two dimensions for easier visualization
reduced_embeddings = UMAP(
    n_components=2, min_dist=0.0, metric="cosine",
random_state=42
# Train the UMAP model and transform the embeddings into 2 dimensions.
).fit_transform(embeddings)
# Create dataframe with column x and y
df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
# Add a new column containing the title of each research paper.
df["title"] = titles
# Add another column containing the clusters assigned by the HDBSCAN
df["cluster"] = [str(c) for c in clusters]
# Select outliers and non-outliers (clusters)
#Outlier = An unusual or isolated data point that is far away from the rest of the data.
clusters_df= df.loc[df.cluster != "-1", :]
outliers_df = df.loc[df.cluster == "-1", :]

In [ ]:
#Import pyplot module form matplotlib
# Matplotlib  is a library that is used to generate graphs
import matplotlib.pyplot as plt
# Plot outliers and non-outliers separately
plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2,c="grey")
plt.axis("on")

In [ ]:
#Plot non-outliers
plt.scatter(clusters_df.x, clusters_df.y,
c=clusters_df.cluster.astype(int),
alpha=0.6, s=2, cmap="tab20b"
)

In [ ]:
# Import Bertopic class from bertopic library
from bertopic import BERTopic
# Train our model with our previously defined models
topic_model = BERTopic(
    # load the previously loaded embedding_model
    embedding_model=embedding_model,
    #load the umap_model
    umap_model=umap_model,
    # load the hdbscan_model
    hdbscan_model=hdbscan_model,
  # display the progress bar while training
verbose=
True
# Train the BERTopic model.
# abstracts = The original text documents. Embeddings =The embeddings already generated from the abstracts.
).fit(abstracts, embeddings)

In [ ]:
# shows the full information about topic_model
topic_model.get_topic_info()

In [ ]:
# Display the keywords that describe Topic 0
topic_model.get_topic(0)

In [ ]:
# Search for topics that are most similar to the phrase "topic modeling".
topic_model.find_topics("topic modeling")

In [ ]:
# Display the keyword that describe topic 22
topic_model.get_topic(22)

In [ ]:
# Find the index of the topic "BERTopic: Neural topic modeling with a class-based TF-IDF procedure"
topic_model.topics_[titles.index("BERTopic: Neural topic modeling with a class-based TF-IDF procedure")]

In [ ]:
# Create a interactive visualization of all documents
fig = topic_model.visualize_documents(
    # Pass the titles of all research papers.
    # When you hover over a point in the graph,its corresponding paper title will be displayed.
    list(titles),
    # load the reduced embeddings (2 dimensional )
    reduced_embeddings=reduced_embeddings,

    # Set the width of the interactive figure
    width=1200,
    # Hide text labels on the graph.
    hide_annotations=True
)
# Update fonts of legend for easier visualization
fig.update_layout(font=dict(size=16))

In [ ]:
# Visualize barchart with ranked keywords
topic_model.visualize_barchart()
# Visualize relationships between topics
topic_model.visualize_heatmap(n_clusters=30)
# Visualize the potential hierarchical structure of topics
topic_model.visualize_hierarchy()

In [ ]:
# Import deepcopy from Python's copy module.
# deepcopy creates a completely independent copy of an object.
from copy import deepcopy
# Save original representations
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
# create a function to to compare the original and updated topic keywords
def topic_differences(model, original_topics, nr_topics=5):
# Show the differences in topic representations between two models """
# create a empty dataframe with 3 columns
  df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
  for topic in range(nr_topics):
    # Extract top 5 words per topic per model
    # Get the original topic keywords.
    # zip(*...) separates words and scores into two lists.
    og_words = " | ".join(list(zip(*original_topics[topic]))
    [0][:5])
    # get the updated topic keywords
    new_words = " | ".join(list(zip(*model.get_topic(topic)))
    [0][:5])
    # add one new row to the dataframe
    df.loc[len(df)] = [topic, og_words, new_words]
    # returns the dataframe
    return df

In [ ]:
# Import KeyBERTInspired representaion model  from bertopic
# This model improves the topic keywords by selecting words that better represent the meaning of each topic.
from bertopic.representation import KeyBERTInspired
# Create an instance of the KeyBERTInspired representation model
representation_model = KeyBERTInspired()
# Update our topic representations using KeyBERTInspired
topic_model.update_topics(abstracts,
representation_model=representation_model)
# Show topic differences
topic_differences(topic_model, original_topics)

In [ ]:
# Import the Maximal Marginal Relevance (MMR) representation model.
# MMR improves topic keywords by selecting words that are  both relevant to the topic and different from each other.
from bertopic.representation import MaximalMarginalRelevance

# Create an MMR representation model.
# diversity controls how different the selected keywords should be.
# diversity = 0.2
# Low diversity = More focus on highly relevant words Some similar words may still appear
representation_model = MaximalMarginalRelevance(diversity=0.2)

# Update the topic representations using the MMR model.
topic_model.update_topics(
    abstracts,
    representation_model=representation_model
)

# Compare the original topic keywords with the
# updated MMR-generated keywords.
topic_differences(topic_model, original_topics)

In [ ]:
# Import pipeline frommtransformers
# A pipeline makes it easy to use pretrained models for different tasks
from transformers import pipeline
# Import TextGeneration representation model from BERTopic
from bertopic.representation import TextGeneration
# input prompt
prompt = """I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: '[KEYWORDS]'.
Based on the documents and keywords, what is this topic about?"""

# Load the FLAN-T5 Small language model using a text-to-text pipeline.
# This model reads the prompt and generates a descriptive topic label.
generator = pipeline("text2text-generation",model="google/flan-t5-small")

# Create a TextGeneration representation model.
representation_model = TextGeneration(

    # Use the FLAN-T5 generator
    generator,

    # Use the custom prompt defined previously
    prompt=prompt,

    # Use the first 50 words from each document when building the prompt.
    doc_length=50,

    # Split the documents into words using whitespace.
    tokenizer="whitespace"
)

# Update the topic representations using the FLAN-T5 model.
# This does NOT change the topics themselves. It only generates better descriptions (labels) for each topic.
topic_model.update_topics(
    abstracts,
    representation_model=representation_model
)

# Compare the original topic keywords with
# the new LLM-generated topic representations.
topic_differences(topic_model, original_topics)

# Below  code requires a paid OpenAI API key to run

In [ ]:
# Import the OpenAI Python library.
# This library allows us to communicate with OpenAI models through the API.
import openai

# Import the OpenAI representation model from BERTopic.
# This allows BERTopic to use GPT to generate better topic labels.
from bertopic.representation import OpenAI

# Create a prompt template.
prompt = """
I have a topic that contains the following documents:

[DOCUMENTS]

The topic is described by the following keywords:

[KEYWORDS]

Based on the information above, extract a short topic label in the following format:

topic: <short topic label>
"""

# Create an OpenAI client.
# Replace "YOUR_API_KEY" with your own API key.
client = openai.OpenAI(
    api_key="YOUR_API_KEY"
)

representation_model = OpenAI(
    client,
    model="gpt-3.5-turbo",
    nr_docs=2,
    exponential_backoff=True,
    chat=True,
    prompt=prompt
)


In [ ]:
# Update the topic representations.
# It only changes the labels of existing topics.
topic_model.update_topics(
    abstracts,
    representation_model=representation_model
)

# Compare the original topic keywords
# with the new GPT-generated topic labels.
topic_differences(topic_model, original_topics)

In [ ]:
# Visualize topics and documents
fig = topic_model.visualize_document_datamap(
    titles,
    topics=list(range(20)),
    reduced_embeddings=reduced_embeddings,
    width=1200,
    label_font_size=11,
    label_wrap_width=20,
    use_medoids=
True
,
)
